Data Cleaning, sub-question 2

In [15]:
# Loading python libraries
import pandas as pd
import os
import numpy as np
## This is a package leanred in CEg for curvefitting data
from scipy.optimize import curve_fit

Loading in Merged CSV file as a dataframe

In [16]:
# Define full file path
file_path = r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\Project data_Freight\merged_eurostat_clean_V2.csv"

# Load dataset
df = pd.read_csv(file_path)

# Select relevant columns
selected_columns = ["geo", "TIME_PERIOD", "Network_length_KM", "Consignment_full_train_load_THS_T", "Consignment_full_wagon_load_THS_T", "Consignment_total_THS_T"]

# Create new DataFrame
df_subset = df[selected_columns]

# Keep only specific countries determined to be usuable by visual inspection
countries_to_keep = ["CH", "DE", "IT", "PL", "SE", "SI", "SK"]

df_subset = df_subset[df["geo"].isin(countries_to_keep)].copy()

# Check
print(df_subset.head())

# Rename
df_merged = df_subset.copy()


   geo  TIME_PERIOD  Network_length_KM  Consignment_full_train_load_THS_T  \
61  CH         2008             5146.0                            37705.0   
62  CH         2009             5188.0                            35765.0   
63  CH         2010             5124.0                            39125.0   
64  CH         2011             5140.0                            40349.0   
65  CH         2012             5147.0                            34013.0   

    Consignment_full_wagon_load_THS_T  Consignment_total_THS_T  
61                            25631.0                  63336.0  
62                            20875.0                  56640.0  
63                            21293.0                  60418.0  
64                            20766.0                  61114.0  
65                            21593.0                  55607.0  


Create helper functions for inter and extrapolating missing data in specified collumns for each geopolitical entity

In [17]:
def fill_with_curvefit(df, col_name, time_col="TIME_PERIOD", group_col="geo", uncertainty_col=None):

    df_filled = df.copy().sort_values(by=[group_col, time_col])

    def linear_func(x, a, b):
        return a * x + b

    filled_groups = []

    for country, group in df_filled.groupby(group_col):
        group = group.copy()
        y = group[col_name].values
        x = group[time_col].values

        mask_valid = ~np.isnan(y)
        if mask_valid.sum() < 2:
            filled_groups.append(group)
            continue

        x_valid = x[mask_valid]
        y_valid = y[mask_valid]
        x_centered = x_valid - x_valid.mean()
        x_all_centered = x - x_valid.mean()

        try:
            popt, pcov = curve_fit(linear_func, x_centered, y_valid)
        except Exception as e:
            print(f"Skipping {country} due to fitting error: {e}")
            filled_groups.append(group)
            continue

        perr = np.sqrt(np.diag(pcov))

        y_pred = linear_func(x_all_centered, *popt)
        y_err = np.sqrt((x_all_centered * perr[0])**2 + perr[1]**2)

        nan_mask = np.isnan(y)
        group.loc[nan_mask, col_name] = y_pred[nan_mask]

        # Use provided uncertainty_col name or a default
        ucol = uncertainty_col or f"uncertainty_estimate_{col_name}"
        group[ucol] = np.where(nan_mask, y_err, np.nan)

        filled_groups.append(group)

    df_result = pd.concat(filled_groups, ignore_index=True)
    print(f" Filled missing values in '{col_name}'.")
    return df_result


Inter-/Extrapolate Network length data

In [18]:
# === Apply to Network_length_KM ===
df_merged = fill_with_curvefit(df_merged, "Network_length_KM", uncertainty_col= "uncertainty_estimate_NL")

# Check result
print(df_merged[df_merged["Network_length_KM"].isna()])

 Filled missing values in 'Network_length_KM'.
Empty DataFrame
Columns: [geo, TIME_PERIOD, Network_length_KM, Consignment_full_train_load_THS_T, Consignment_full_wagon_load_THS_T, Consignment_total_THS_T, uncertainty_estimate_NL]
Index: []


Inter-/Extrapolate Full train data

In [19]:
# === Apply to Consignment_full_train_load_THS_T ===
df_merged = fill_with_curvefit(df_merged, "Consignment_full_train_load_THS_T", uncertainty_col= "uncertainty_estimate_FULL_TR")

# Check if any NaNs remain
print(df_merged[df_merged["Consignment_full_train_load_THS_T"].isna()])


 Filled missing values in 'Consignment_full_train_load_THS_T'.
Empty DataFrame
Columns: [geo, TIME_PERIOD, Network_length_KM, Consignment_full_train_load_THS_T, Consignment_full_wagon_load_THS_T, Consignment_total_THS_T, uncertainty_estimate_NL, uncertainty_estimate_FULL_TR]
Index: []


Inter-/Extrapolate Full_wagon consignment data

In [20]:
# === Apply to Consignment_full_wagon_load_THS_T ===
df_merged = fill_with_curvefit(df_merged, col_name="Consignment_full_wagon_load_THS_T", uncertainty_col= "uncertainty_estimate_FULL_WG")

# Check if any NaNs remain
print(df_merged[df_merged["Consignment_full_wagon_load_THS_T"].isna()])

 Filled missing values in 'Consignment_full_wagon_load_THS_T'.
Empty DataFrame
Columns: [geo, TIME_PERIOD, Network_length_KM, Consignment_full_train_load_THS_T, Consignment_full_wagon_load_THS_T, Consignment_total_THS_T, uncertainty_estimate_NL, uncertainty_estimate_FULL_TR, uncertainty_estimate_FULL_WG]
Index: []


Inter-/Extrapolate Total consignment data

In [21]:
# === Apply to Consignment_total_THS_T ===
df_merged = fill_with_curvefit(df_merged, col_name="Consignment_total_THS_T", uncertainty_col= "uncertainty_estimate_TOTAL")

# Check that all NaNs remain
print(df_merged[df_merged["Consignment_total_THS_T"].isna()])

 Filled missing values in 'Consignment_total_THS_T'.
Empty DataFrame
Columns: [geo, TIME_PERIOD, Network_length_KM, Consignment_full_train_load_THS_T, Consignment_full_wagon_load_THS_T, Consignment_total_THS_T, uncertainty_estimate_NL, uncertainty_estimate_FULL_TR, uncertainty_estimate_FULL_WG, uncertainty_estimate_TOTAL]
Index: []


Save current curve_fitted data

In [22]:
# Get the current working directory (where your notebook/script is)
current_dir = os.getcwd()

# Define output filename
output_path = os.path.join(current_dir, "df_merged_cleaned.csv")

# Export to CSV
df_merged.to_csv(output_path, index=False)

print(f" Data successfully exported to:\n{output_path}")

 Data successfully exported to:
c:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\df_merged_cleaned.csv


Add population stats

In [23]:
dir_pathing = r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\tps00001__custom_18702395_linear_2_0.csv"

# Load population data
pop = pd.read_csv(dir_pathing)

# Keep only relevant columns and rename
pop = pop[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"OBS_VALUE": "population"})

# Merge using geo and TIME_PERIOD
df_merged = df_merged.merge(pop, on=["geo", "TIME_PERIOD"], how="left")

# Check merge results
print(f"Rows merged: {len(df_merged)}")
print(f"Missing population entries: {df_merged['population'].isna().sum()}")

Rows merged: 119
Missing population entries: 42


Inter-/Extrapolate population data

In [24]:
# === Apply to population ===
df_merged = fill_with_curvefit(df_merged, col_name="population", uncertainty_col="uncertainty_estimate_POP")

#  Check if any NaNs remain
missing_after = df_merged["population"].isna().sum()
print(f"Remaining missing population entries: {missing_after}")

 Filled missing values in 'population'.
Remaining missing population entries: 0


Adding Area data for specific countries in merged set

In [25]:
# Load Eurostat area dataset
area = pd.read_csv(r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\reg_area3__custom_18702514_linear_2_0.csv")

# Keep only relevant columns
area = area[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"OBS_VALUE": "land_area_km2"})

# Merge area into the *existing in-memory df_merged*
df_merged = df_merged.merge(area, on=["geo", "TIME_PERIOD"], how="left")

# Fill missing area values (using 2014 value only for NaNs before 2014)
def fill_area_before_2014(group):
    val_2014 = group.loc[group["TIME_PERIOD"] == 2014, "land_area_km2"]
    if not val_2014.empty:
        value = val_2014.iloc[0]
        mask = (group["TIME_PERIOD"] < 2014) & (group["land_area_km2"].isna())
        group.loc[mask, "land_area_km2"] = value
    return group

df_merged = df_merged.groupby("geo", group_keys=False).apply(fill_area_before_2014)

#  Save updated version (now with population + uncertainty + area)
df_merged.to_csv(r"C:\Users\youri\OneDrive\Desktop\TIL Programming\6020 Group project\df_cleaned_population_area.csv",index=False)

C:\Users\youri\AppData\Local\Temp\ipykernel_26592\3194539748.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_merged = df_merged.groupby("geo", group_keys=False).apply(fill_area_before_2014)


In [ ]:
# Display the entire DataFrame
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df_merged)

,geo,TIME_PERIOD,Network_length_KM,Consignment_full_train_load_THS_T,Consignment_full_wagon_load_THS_T,Consignment_total_THS_T,uncertainty_estimate_NL,uncertainty_estimate_FULL_TR,uncertainty_estimate_FULL_WG,uncertainty_estimate_TOTAL,population,uncertainty_estimate_POP,land_area_km2
0,CH,2008,5146.000000,37705.000000,25631.000000,63336.000000,NaN,NaN,NaN,NaN,7.712250e+06,28266.381550,41289.0
1,CH,2009,5188.000000,35765.000000,20875.000000,56640.000000,NaN,NaN,NaN,NaN,7.787544e+06,25901.857604,41289.0
2,CH,2010,5124.000000,39125.000000,21293.000000,60418.000000,NaN,NaN,NaN,NaN,7.862839e+06,23558.913893,41289.0
3,CH,2011,5140.000000,40349.000000,20766.000000,61114.000000,NaN,NaN,NaN,NaN,7.938134e+06,21244.691464,41289.0
4,CH,2012,5147.000000,34013.000000,21593.000000,55607.000000,NaN,NaN,NaN,NaN,8.013428e+06,18969.704850,41289.0
5,CH,2013,5181.000000,36401.000000,23104.000000,59505.000000,NaN,NaN,NaN,NaN,8.088723e+06,16749.948774,41289.0
6,CH,2014,5187.000000,38197.000000,22149.000000,60345.000000,NaN,NaN,NaN,NaN,8.139631e+06,NaN,41289.0
7,CH,2015,5205.000000,39475.000000,21357.000000,60832.000000,NaN,NaN,NaN,NaN,8.237666e+06,NaN,41289.0
8,CH,2016,5249.000000,40040.000000,21108.000000,61148.000000,NaN,NaN,NaN,NaN,8.327126e+06,NaN,41289.0
9,CH,2017,5251.000000,44459.000000,14638.000000,59097.000000,NaN,NaN,NaN,NaN,8.419550e+06,NaN,41289.0
